# GLiClass 퓨샷 파인튜닝 2차 시범: 학습량 조절 + 쏠림 진단

1차 결과: k=1~5 모두 거의 전부 defense-evasion으로 찍음, 학습 전술 정확도 0, 판별력 0.

**이번에 확인할 것 3가지**

1. **학습이 실제로 일어나나**: 학습에 쓴 줄 자체를 맞히는지 (학습 데이터 정확도), 학습 손실이 줄어드는지
2. **학습량 문제인가**: 예시 수 k는 그대로(1, 5), **에폭(10/30/100)과 학습률(2e-5/1e-4)만** 바꿔서 비교. 예시 수가 그대로라 퓨샷 조건은 유지됨
3. **숨긴 라벨 쏠림인가**: 평가를 세 가지로 나눠 본다
    - 전체 평가: 라벨 11개 전부 놓고 1등 (1차와 같음)
    - 나눈 평가: 학습 전술 윈도우는 학습 라벨 8개 중에서만, 숨긴 전술 윈도우는 숨긴 라벨 3개 + benign 중에서만 고름 → 두 그룹끼리 경쟁 안 함
    - 정답 점수: 1등과 상관없이 정답 라벨의 sigmoid 확률이 오르는지

조합: k 2개 × 에폭 3개 × 학습률 2개 × 시드 2개 = 24번 + 기준선. **약 30~40분** (T4 GPU).

준비: 런타임 유형 T4 GPU로 바꾼 뒤 `cloudtrail_windows.csv` 업로드.

## 1. 설치

In [ ]:
COMMIT = '6a00f8067648f25aa0f4f477d167c5f391cb4165'
!pip -q install "git+https://github.com/Knowledgator/GLiClass.git@{COMMIT}"
!git clone -q https://github.com/Knowledgator/GLiClass.git gliclass_repo && cd gliclass_repo && git checkout -q {COMMIT}
!ls gliclass_repo/train.py

설치 뒤 다음 칸에서 import 에러가 나면: 런타임 > 세션 다시 시작 후 **2번부터** 실행.

## 2. 설정

In [ ]:
import os, re, json, random, subprocess, shutil, time, itertools
import numpy as np, pandas as pd, torch

MODEL = 'knowledgator/gliclass-edge-v3.0'
K_LIST = [1, 5]
EPOCHS_LIST = [10, 30, 100]
LR_LIST = [2e-5, 1e-4]
SEEDS = [0, 1]
BATCH, MAX_LEN = 8, 2048
BENIGN_POOL_N, BENIGN_TEST_N = 50, 500
HELD_OUT = ['defense-evasion', 'exfiltration', 'discovery']

LABELS = {
    'initial-access': 'The adversary is trying to get into your network.',
    'execution': 'The adversary is trying to run malicious code.',
    'persistence': 'The adversary is trying to maintain their foothold.',
    'privilege-escalation': 'The adversary is trying to gain higher-level permissions.',
    'defense-evasion': 'The adversary is trying to avoid being detected.',
    'credential-access': 'The adversary is trying to steal account names and passwords.',
    'discovery': 'The adversary is trying to figure out your environment.',
    'lateral-movement': 'The adversary is trying to move through your environment.',
    'exfiltration': 'The adversary is trying to steal data.',
    'impact': 'The adversary is trying to manipulate, interrupt, or destroy your systems and data.',
    'benign': 'Normal administrative, automated or user activity with no adversary.',
}
TEXT2LABEL = {v: k for k, v in LABELS.items()}
SEEN = [t for t in LABELS if t not in HELD_OUT]          # 학습 라벨 8개 (benign 포함)
HELD_SET = HELD_OUT + ['benign']                          # 숨긴 전술 평가용 라벨 4개
print('GPU:', torch.cuda.is_available())

## 3. 데이터 나누기 (1차와 동일)

In [ ]:
df = pd.read_csv('cloudtrail_windows.csv').drop_duplicates(['source', 'text', 'labels']).reset_index(drop=True)
df['label'] = df['labels'].str.split('|').str[0]
pool = df[(df.source == 'datadog') & (~df.label.isin(HELD_OUT))]
inv = df[df.source == 'invictus']
benign = inv[inv.label == 'benign'].sample(frac=1, random_state=42)
benign_pool = benign.iloc[:BENIGN_POOL_N]
test = pd.concat([inv[inv.label != 'benign'], benign.iloc[BENIGN_POOL_N:BENIGN_POOL_N + BENIGN_TEST_N]])
test = test[~test.text.isin(df[df.source == 'datadog'].text)].reset_index(drop=True)
seen_test = test[test.label.isin(SEEN)].reset_index(drop=True)                   # 학습 전술 공격 + benign
held_test = test[test.label.isin(HELD_SET)].reset_index(drop=True)               # 숨긴 전술 공격 + benign
print('시험:', test.label.value_counts().to_dict())
print('나눈 평가 - 학습 그룹', len(seen_test), '/ 숨긴 그룹', len(held_test))

## 4. 학습 데이터 만들기 (1차와 동일: 라벨마다 서로 다른 실행에서 k개)

In [ ]:
def pick(group, k, rng):
    runs = list(group.groupby('run_id')); rng.shuffle(runs)
    picked, i = [], 0
    while len(picked) < min(k, len(group)):
        _, g = runs[i % len(runs)]
        rest = g[~g.index.isin([p.name for p in picked])]
        if len(rest):
            picked.append(rest.sample(1, random_state=rng.randint(0, 10**6)).iloc[0])
        i += 1
    return picked

def make_train(k, seed):
    rng = random.Random(seed)
    seen_texts = [LABELS[t] for t in SEEN]
    rows = []
    for tactic, g in pool.groupby('label'):
        for r in pick(g, k, rng):
            rows.append({'text': r.text, 'all_labels': seen_texts, 'true_labels': [LABELS[tactic]]})
    for _, r in benign_pool.sample(k, random_state=seed).iterrows():
        rows.append({'text': r.text, 'all_labels': seen_texts, 'true_labels': [LABELS['benign']]})
    rng.shuffle(rows)
    return rows
print({k: len(make_train(k, 0)) for k in K_LIST})

## 5. 평가 함수

라벨별 sigmoid 확률을 전부 받아서 여러 방식으로 계산한다.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from gliclass import GLiClassModel, ZeroShotClassificationPipeline

agnews = load_dataset('fancyzhx/ag_news', split='test').shuffle(seed=0).select(range(500))
AG_LABELS = ['World', 'Sports', 'Business', 'Sci/Tech']

def scores(pipe, texts, label_keys):
    texts_l = [LABELS[t] for t in label_keys]
    out = pipe(list(texts), texts_l, threshold=0.0, batch_size=16)
    return [{TEXT2LABEL[x['label']]: x['score'] for x in r} for r in out]

def top1(sc):
    return [max(s, key=s.get) for s in sc]

def sep(y, p):
    att = [t != 'benign' for t in y]
    rec = np.mean([q != 'benign' for q, a in zip(p, att) if a])
    fpr = np.mean([q != 'benign' for q, a in zip(p, att) if not a])
    return rec, fpr

def evaluate(path, train_rows=None):
    model = GLiClassModel.from_pretrained(path)
    tok = AutoTokenizer.from_pretrained(path)
    pipe = ZeroShotClassificationPipeline(model, tok, classification_type='multi-label',
                                          device='cuda:0', max_length=MAX_LEN, progress_bar=False)
    r = {}
    # (가) 전체 평가: 라벨 11개
    sc = scores(pipe, test.text, list(LABELS)); y = test.label.tolist(); p = top1(sc)
    r['all_seen_acc'] = np.mean([q == t for q, t in zip(p, y) if t not in HELD_OUT and t != 'benign'])
    r['all_held_acc'] = np.mean([q == t for q, t in zip(p, y) if t in HELD_OUT])
    rec, fpr = sep(y, p); r['all_sep'] = rec - fpr
    r['top_pred'] = pd.Series(p).value_counts().head(2).to_dict()
    # (다) 정답 점수: 정답 라벨 확률 평균 vs 오답 라벨 확률 평균
    r['true_score_seen'] = np.mean([s[t] for s, t in zip(sc, y) if t in SEEN])
    r['true_score_held'] = np.mean([s[t] for s, t in zip(sc, y) if t in HELD_OUT])
    r['wrong_score'] = np.mean([v for s, t in zip(sc, y) for l, v in s.items() if l != t])
    r['mean_score_seen_labels'] = np.mean([s[l] for s in sc for l in SEEN])
    r['mean_score_held_labels'] = np.mean([s[l] for s in sc for l in HELD_OUT])
    # (나) 나눈 평가
    ps = top1(scores(pipe, seen_test.text, SEEN)); ys = seen_test.label.tolist()
    r['split_seen_acc'] = np.mean([q == t for q, t in zip(ps, ys) if t != 'benign'])
    rec, fpr = sep(ys, ps); r['split_seen_recall'], r['split_seen_fpr'], r['split_seen_sep'] = rec, fpr, rec - fpr
    ph = top1(scores(pipe, held_test.text, HELD_SET)); yh = held_test.label.tolist()
    r['split_held_acc'] = np.mean([q == t for q, t in zip(ph, yh) if t != 'benign'])
    rec, fpr = sep(yh, ph); r['split_held_sep'] = rec - fpr
    # 학습 데이터 자체를 맞히나
    if train_rows:
        pt = top1(scores(pipe, [x['text'] for x in train_rows], SEEN))
        r['train_acc'] = np.mean([q == TEXT2LABEL[x['true_labels'][0]] for q, x in zip(pt, train_rows)])
    # 원래 실력
    ag = pipe(agnews['text'], AG_LABELS, threshold=0.0, batch_size=16)
    r['agnews_acc'] = np.mean([AG_LABELS.index(max(a, key=lambda x: x['score'])['label']) == b
                               for a, b in zip(ag, agnews['label'])])
    del model, pipe; torch.cuda.empty_cache()
    return r

def show(r):
    return {k: (round(float(v), 3) if isinstance(v, (float, np.floating)) else v) for k, v in r.items()}

base = evaluate(MODEL)
show(base)

## 6. 실행: k × 에폭 × 학습률 × 시드

학습 손실은 `train.py` 출력에서 처음과 마지막 값을 뽑아 기록한다. 매 실행마다 `pilot2_results.csv`에 저장.

In [ ]:
arch = GLiClassModel.from_pretrained(MODEL).config.architecture_type
results = [{'k': 0, 'epochs': 0, 'lr': 0, 'seed': 0, **base}]

def losses(stdout):
    v = [float(x) for x in re.findall(r"'loss': '?([0-9.]+)", stdout)]
    return (v[0], v[-1]) if v else (np.nan, np.nan)

for k, ep, lr, seed in itertools.product(K_LIST, EPOCHS_LIST, LR_LIST, SEEDS):
    rows = make_train(k, seed)
    json.dump(rows, open('train.json', 'w'))
    out = f'runs/k{k}_e{ep}_lr{lr}_s{seed}'
    steps = max(1, len(rows) // BATCH)
    cmd = ['python', 'gliclass_repo/train.py', '--model_name', MODEL,
           '--data_path', 'train.json', '--val_data_path', 'train.json',
           '--save_path', out, '--problem_type', 'multi_label_classification', '--architecture_type', arch,
           '--encoder_lr', str(lr), '--others_lr', str(lr),
           '--num_epochs', str(ep), '--batch_size', str(BATCH), '--max_length', str(MAX_LEN),
           '--focal_loss_alpha', '0.5', '--focal_loss_gamma', '2',
           '--num_workers', '0', '--save_steps', '1000000', '--logging_steps', str(steps)]
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout[-2000:], proc.stderr[-3000:]); raise SystemExit(f'학습 실패 {out}')
    l0, l1 = losses(proc.stdout + proc.stderr)
    m = evaluate(os.path.join(out, 'final_model'), rows)
    shutil.rmtree(out, ignore_errors=True)
    results.append({'k': k, 'epochs': ep, 'lr': lr, 'seed': seed, 'n_train': len(rows),
                    'loss_first': l0, 'loss_last': l1, **m})
    pd.DataFrame(results).to_csv('pilot2_results.csv', index=False)
    print(f'k={k} ep={ep:3d} lr={lr:.0e} s={seed} ({time.time()-t0:.0f}s) loss {l0:.3f}->{l1:.3f} | '
          f'train_acc {m["train_acc"]:.2f} | 나눈평가 학습전술 {m["split_seen_acc"]:.2f} 숨긴전술 {m["split_held_acc"]:.2f} '
          f'판별력 {m["split_seen_sep"]:.2f} | 전체평가 1등 {m["top_pred"]} | ag_news {m["agnews_acc"]:.3f}')

## 7. 요약표

In [ ]:
res = pd.read_csv('pilot2_results.csv')
cols = ['loss_last', 'train_acc', 'split_seen_acc', 'split_seen_sep', 'split_held_acc',
        'all_seen_acc', 'all_held_acc', 'true_score_seen', 'true_score_held', 'wrong_score',
        'mean_score_seen_labels', 'mean_score_held_labels', 'agnews_acc']
summary = res.groupby(['k', 'epochs', 'lr'])[cols].mean().round(3)
summary

## 8. 쏠림 확인 그래프

학습 라벨(SEEN)과 숨긴 라벨의 평균 점수를 비교. 숨긴 라벨 점수가 학습 라벨보다 높게 남아 있으면 쏠림이 원인이라는 증거.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, k in zip(axes, K_LIST):
    sub = res[(res.k == k) | (res.k == 0)].copy()
    sub['setting'] = sub.apply(lambda r: 'before' if r.k == 0 else f"ep{int(r.epochs)}\nlr{r.lr:.0e}", axis=1)
    g = sub.groupby('setting', sort=False)[['mean_score_seen_labels', 'mean_score_held_labels', 'true_score_seen']].mean()
    g.plot(kind='bar', ax=ax, rot=0)
    ax.set(title=f'k={k}: average sigmoid score', ylabel='score', ylim=(0, 1))
    ax.legend(['seen labels (avg)', 'held-out labels (avg)', 'correct label (seen windows)'], fontsize=8)
fig.tight_layout(); fig.savefig('pilot2_bias.png', dpi=200)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for (k, lr), g in res[res.k > 0].groupby(['k', 'lr']):
    gm = g.groupby('epochs')[['train_acc', 'split_seen_acc', 'agnews_acc']].mean()
    ax.plot(gm.index, gm.train_acc, marker='o', label=f'train acc k={k} lr={lr:.0e}')
    ax.plot(gm.index, gm.split_seen_acc, marker='s', ls='--', label=f'test seen acc k={k} lr={lr:.0e}')
ax.axhline(base['agnews_acc'], color='grey', lw=.8, ls=':', label='ag_news before')
ax.set(xscale='log', xlabel='epochs', ylabel='accuracy', ylim=(0, 1.05), title='Does it learn? (train vs test)')
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=.3); fig.tight_layout(); fig.savefig('pilot2_learning.png', dpi=200)

## 9. 결과 읽는 법

| 관찰 | 뜻 |
|---|---|
| train_acc가 1에 가까워지고 loss가 줄어듦 | 학습은 일어남 |
| train_acc는 높은데 split_seen_acc는 낮음 | 학습 예시만 외움, 다른 계정 로그로 일반화 안 됨 |
| 숨긴 라벨 평균 점수 > 학습 라벨 평균 점수 | 쏠림이 원인 → 나눈 평가를 기준으로 봐야 함 |
| split_held_acc가 기준선보다 오름 | 처음 보는 전술에도 도움 (제로샷 유지/향상) |
| agnews_acc가 기준선보다 크게 떨어짐 | 원래 실력 망각 → 리플레이/EWC 필요 |
| 에폭/학습률을 올려도 train_acc가 안 오름 | 학습 설정 자체 문제 (라벨 형식, 손실 등) 점검 필요 |

`pilot2_results.csv`, `pilot2_bias.png`, `pilot2_learning.png`를 보내주면 같이 해석.